# AI Personal Coach — OULAD EDA & Baseline Churn Model Notebook
**Yazar**: Kişi 3 — Data Lead (EDA)
**Proje**: AI Personal Coach (LGS/YKS Veli Abonelik Retention Modeli)
**Amaç**: Open University Learning Analytics Dataset (OULAD) ve sentetik dikkat/anket proxy verisi kullanarak 90 günlük veli retention/churn risk modelini eğitmek, 5 öğrenci alt segmentini analiz etmek ve Kişi 5 (Tech Review) için karşılaştırmalı baseline model metriklerini üretmek.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, log_loss, roc_curve
import lightgbm as lgb

# Veri yükleme
df = pd.read_csv('../oulad_synthetic_processed.csv')
print(f"Veri seti boyutu: {df.shape}")
df.head()

## 1. Betimsel İstatistikler ve Özellik Özetleri
Aşağıdaki tabloda değişkenlerin ortalama, standart sapma, min ve maks değerleri görüntülenmektedir.

In [2]:
df.describe().T

## 2. Öğrenci Alt Segment Analizi (5 Persona)
Proje kapsamında tanımlanan 5 alt segment:
1. **Başlayamayan** (Ders başlatma stresi, erteleme)
2. **Yarıda Bırakan** (Sürdürülebilirlik kaybı)
3. **Telefonla Dağılan** (Odak bloğunda app > 10 dk)
4. **Kaygıyla Erteleyen** (Deneme öncesi ders kaçırma)
5. **Geceye Kayan** (Gündüz atıllığı, gece yığılma)

In [3]:
segment_summary = df.groupby('sub_segment').agg(
    Ogrenci_Sayisi=('student_id', 'count'),
    Ort_VLE_Tiklama=('vle_total_clicks', 'mean'),
    Ort_Odak_Suresi=('avg_focus_duration_mins', 'mean'),
    Ort_Telefon_Uyarisi=('phone_distraction_10min_count', 'mean'),
    Churn_Orani=('churn_90d', 'mean')
).reset_index()
segment_summary['Churn_Orani_%'] = (segment_summary['Churn_Orani'] * 100).round(2)
segment_summary

## 3. Baseline Model Eğitimi & Kişi 5 Handoff Metrikleri
Logistic Regression ve LightGBM modelleri 90 günlük retention (0) / churn (1) hedefi üzerinde eğitilmektedir.

In [4]:
X = df.drop(columns=['student_id', 'exam_type', 'grade', 'sub_segment', 'churn_90d'])
y = df['churn_90d']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
lr_preds = lr.predict(X_test)

# LightGBM
lgb_cls = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1)
lgb_cls.fit(X_train, y_train)
lgb_probs = lgb_cls.predict_proba(X_test)[:, 1]
lgb_preds = lgb_cls.predict(X_test)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'LightGBM Classifier'],
    'ROC-AUC': [roc_auc_score(y_test, lr_probs), roc_auc_score(y_test, lgb_probs)],
    'F1-Score': [f1_score(y_test, lr_preds), f1_score(y_test, lgb_preds)],
    'Precision': [precision_score(y_test, lr_preds), precision_score(y_test, lgb_preds)],
    'Recall': [recall_score(y_test, lr_preds), recall_score(y_test, lgb_preds)],
    'LogLoss': [log_loss(y_test, lr_probs), log_loss(y_test, lgb_probs)]
})
results